In [1]:
import numpy as np
np.bool = np.bool_ 
import pandas as pd
import os
from tqdm import tqdm

import mxnet as mx
from mxnet import gluon
from mxnet import autograd
from mxnet import image

import sys

from sklearn.metrics import matthews_corrcoef

import random

import rasterio
from rasterio.enums import ColorInterp
from rasterio.features import shapes
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

import higra as hg
import torch
import pickle

import itertools

import imageio.v3 as imageio
from skimage.color import label2rgb
import cv2
from skimage.segmentation import mark_boundaries
from skimage.segmentation import find_boundaries

from shapely.geometry import shape
import shapely
from shapely import Polygon, MultiPolygon, orient_polygons
from shapely.wkt import loads
from shapely import ops
from shapely.ops import polygonize

import geopandas as gpd
import pyproj

import ast

import glob

import multiprocessing

In [2]:
data = pd.read_csv('inference_data/inference_data.csv')
image_filenames = data[data["num_cultivated_pixels"] > 0]["image_filename"].tolist() # images that have at least one cultivated pixel

# Create map product with conformal confidence set

In [3]:
# https://stackoverflow.com/questions/75756119/concatenate-10-000-csv-files-in-a-directory-using-python-pandas-too-slow
filenames = [f'inference_data/outputs/field_predictions/{image_filename[-9:-4]}.csv' for image_filename in image_filenames]
with open("inference_data/outputs/field_predictions_final.csv", "w") as output:
  first = True
  for filename in tqdm(filenames, smoothing=0):
    with open(filename, "r") as inputfile:
      for no, line in enumerate(inputfile, 1):
        if no == 1 and not first:
          continue
        first = False
        output.write(line)

100%|██████████| 52409/52409 [3:04:02<00:00,  4.75it/s]  


In [4]:
# Count number of predicted fields
total_fields = 0
for chunk in tqdm(pd.read_csv("inference_data/outputs/field_predictions_final.csv", chunksize=100000)):
    total_fields += len(chunk)
print(total_fields)

646it [28:16,  2.63s/it]

64531292


In [3]:
# Split map product into 2 files to get around Google Earth Engine file size limit
i = 0
for df in tqdm(pd.read_csv("inference_data/outputs/field_predictions_final.csv", chunksize=1000000)):
    header = (i == 0 or i == 32)
    if i < 32:
        output_filename = 'inference_data/outputs/field_predictions_final_1.csv'
    else:
        output_filename = 'inference_data/outputs/field_predictions_final_2.csv'
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1

65it [1:32:31, 85.40s/it] 


In [5]:
# Clean up predictions (remove collapsed part of polygons)
def get_final_geometry(x):
    x = loads(x)
    if x.geom_type == 'GeometryCollection':
        polygons = list(polygonize(x.geoms))
        if len(polygons) == 1:
            return polygons[0]
        elif len(polygons) > 1:
            return MultiPolygon(polygons)
        else:
            return x
    else:
        return x
    
i = 0
output_filename = 'inference_data/outputs/field_predictions_FINAL_1.csv'
for df in tqdm(pd.read_csv("inference_data/outputs/field_predictions_final_1.csv", chunksize=1000000)):
    header = (i == 0)
    df['geometry'] = df['geometry'].apply(get_final_geometry)
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1

32it [1:46:44, 200.14s/it]


In [6]:
def get_final_geometry(x):
    x = loads(x)
    if x.geom_type == 'GeometryCollection':
        polygons = list(polygonize(x.geoms))
        if len(polygons) == 1:
            return polygons[0]
        elif len(polygons) > 1:
            return MultiPolygon(polygons)
        else:
            return x
    else:
        return x
    
i = 0
output_filename = 'inference_data/outputs/field_predictions_FINAL_2.csv'
for df in tqdm(pd.read_csv("inference_data/outputs/field_predictions_final_2.csv", chunksize=1000000)):
    header = (i == 0)
    df['geometry'] = df['geometry'].apply(get_final_geometry)
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1

33it [1:55:06, 209.28s/it]


# Create map product for single best threshold (T=0.08)

In [3]:
# https://stackoverflow.com/questions/75756119/concatenate-10-000-csv-files-in-a-directory-using-python-pandas-too-slow
filenames = [f'inference_data/outputs/field_predictions_single_best_threshold/{image_filename[-9:-4]}.csv' for image_filename in image_filenames]
with open("inference_data/outputs/field_predictions_single_best_threshold.csv", "w") as output:
  first = True
  for filename in tqdm(filenames, smoothing=0):
    with open(filename, "r") as inputfile:
      for no, line in enumerate(inputfile, 1):
        if no == 1 and not first:
          continue
        first = False
        output.write(line)

100%|██████████| 52409/52409 [19:42<00:00, 44.30it/s]


In [7]:
def get_final_geometry(x):
    x = loads(x)
    if x.geom_type == 'GeometryCollection':
        polygons = list(polygonize(x.geoms))
        if len(polygons) == 1:
            return polygons[0]
        elif len(polygons) > 1:
            return MultiPolygon(polygons)
        else:
            return x
    else:
        return x
    
i = 0
output_filename = 'inference_data/outputs/field_predictions_single_best_threshold_FINAL.csv'
for df in tqdm(pd.read_csv("inference_data/outputs/field_predictions_single_best_threshold.csv", chunksize=1000000)):
    header = (i == 0)
    df['geometry'] = df['geometry'].apply(get_final_geometry)
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1

17it [1:01:56, 218.61s/it]
